# M12 — PPO: Full 6000-Episode Budget Training Run (Colab T4)

This notebook trains the **PPO (M12)** continuous Gaussian baseline for a full paper-scale budget (**6000 episodes**, converted internally to **1,200,000 total environment steps** with `N_SLOTS=200`), matching PKTD3-TD's Table III `M_EPISODES`.

**Architecture & Training:**
- Continuous action space matching PKTD3-TD: 3D physical velocity $(v, \lambda, \rho)$ via $[-1, 1]^3$ normalized actions
- Separate Actor (Gaussian policy with learnable log-std) and Critic (state-value $V(s)$ network)
- Generalized Advantage Estimation (GAE-Lambda=0.95)
- PPO-Clip surrogate objective (clip_eps=0.2) with entropy bonus (coef=0.01)
- 2,048-step rollouts, 10 update epochs with minibatch size 64

**Workflow pattern:**
1. Cloned fresh to local Colab SSD (`/content/uav_trajectory_rl`) for fast I/O.
2. Checkpoints saved directly to **Google Drive** for persistent safety against disconnects.
3. Checkpoints saved every 50,000 steps (24 total checkpoints, matching the 250-episode cadence of run4).
4. At the end, 30-seed deterministic evaluation is run, checkpoints copied to repo, and pushed to GitHub using Colab's `GITHUB_PAT_TOKEN` secret.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo fresh (local disk) and install

In [ ]:
# Clone repo fresh to local disk with authenticated URL for seamless push at the end
import os
from google.colab import userdata

try:
    github_token = userdata.get('GITHUB_PAT_TOKEN')
except Exception:
    github_token = None

repo_owner = "Krishna200608"
repo_name = "uav_trajectory_rl"

if github_token:
    repo_url = f"https://{github_token}@github.com/{repo_owner}/{repo_name}.git"
    print("Authenticated git URL configured using GITHUB_PAT_TOKEN secret.")
else:
    repo_url = f"https://github.com/{repo_owner}/{repo_name}.git"
    print("WARNING: GITHUB_PAT_TOKEN secret not found. Git push in Cell 8 will require manual auth.")

%cd /content
!rm -rf uav_trajectory_rl
!git clone {repo_url} uav_trajectory_rl
%cd uav_trajectory_rl
!pip install -e . --quiet
!git log --oneline -n 5

## 3. Confirm GPU and check train_ppo.py's actual CLI flags

Do not assume the flags below are exactly right — confirm against this output first.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print()
!python scripts/train_ppo.py --help

## 4. Set the Drive checkpoint path

In [ ]:
import os
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/ppo_run1"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be written to:', DRIVE_CHECKPOINT_DIR)

## 5. Launch the full training run

Budget: `--episodes 6000` automatically sets `total_steps = 1,200,000` (6000 episodes × 200 slots), exactly matching PKTD3-TD's training exposure. With `rollout_length=2048`, this is ~586 rollout collection & update cycles.

Saving checkpoints every `50000` steps (`--checkpoint-every 50000`) produces **24 checkpoints across the run** (matching the 250-episode cadence of `run4` and Dueling DQL).

**This cell will run for a while — let it complete.**

In [ ]:
!python scripts/train_ppo.py \
  --episodes 6000 \
  --seed 0 \
  --rollout-length 2048 \
  --checkpoint-dir "{DRIVE_CHECKPOINT_DIR}" \
  --checkpoint-every 50000 \
  --log-every 50

## 6. Sanity-check the run completed properly

In [ ]:
import glob, os, numpy as np

ckpts = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*.pt"))
print(f'Found {len(ckpts)} checkpoint files in Drive:')
for c in ckpts[-5:]:
    print(' ', os.path.basename(c))

rewards_path = f"{DRIVE_CHECKPOINT_DIR}/episode_rewards.npy"
if os.path.exists(rewards_path):
    r = np.load(rewards_path)
    print(f'\nTotal episodes logged: {len(r)}')
    print(f'First 100 mean reward: {r[:100].mean():.2f}')
    print(f'Last 100 mean reward:  {r[-100:].mean():.2f}')
    print(f'Max episode reward:    {r.max():.2f}')
else:
    print('WARNING: episode_rewards.npy not found -- check the run completed correctly.')

## 7. Behavioral check — arrival rate and entropy trend, not just reward

The 800-episode diagnostic showed PPO covering ~700m of the ~848m diagonal but 0% arrival — check whether the full budget closes that final gap, and whether policy entropy has genuinely decreased (committed policy) rather than staying high (still mostly random).

In [ ]:
import glob, os, torch
import numpy as np
from uav_trajectory_rl.mdp_environment import UAVTrajectoryEnv
from uav_trajectory_rl.baselines.ppo import PPOAgent

# Locate final checkpoint
final_ckpt_candidates = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*final*.pt")) or sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*.pt"))
if not final_ckpt_candidates:
    raise FileNotFoundError(f"No checkpoint files found in {DRIVE_CHECKPOINT_DIR}")
final_ckpt = final_ckpt_candidates[-1]
print('Evaluating checkpoint:', final_ckpt)

env_tmp = UAVTrajectoryEnv(k=10, rng=np.random.default_rng(0))
agent = PPOAgent(state_dim=env_tmp.state_dim)
agent.load(final_ckpt)
agent.actor.eval()
agent.critic.eval()

def rollout(seed, k=10):
    env = UAVTrajectoryEnv(k=k, rng=np.random.default_rng(seed))
    state = env.reset()
    start = env.uav_pos.copy()
    max_dist, done, steps, ep_reward, arrived = 0.0, False, 0, 0.0, False
    while not done and steps < 200:
        # NOTE: select_action_deterministic() internally evaluates the actor Gaussian mean,
        # clips to [-c, c], and unnormalizes to physical (v, lam, rho) directly.
        physical_action = agent.select_action_deterministic(state)
        state, r, done, info = env.step(physical_action)
        max_dist = max(max_dist, float(np.linalg.norm(env.uav_pos - start)))
        ep_reward += r
        steps += 1
        arrived = info.get('arrived', False)
    return max_dist, ep_reward, arrived, steps

dists, rewards, arrivals, steps_taken = [], [], [], []
for seed in range(30):
    d, r, a, s = rollout(seed)
    dists.append(d); rewards.append(r); arrivals.append(a); steps_taken.append(s)
dists = np.array(dists)

print(f'\n30-seed evaluation (deterministic mean action):')
print(f'  Mean max displacement:   {dists.mean():.1f} m')
print(f'  Median max displacement: {np.median(dists):.1f} m')
print(f'  Frac > 50m:              {(dists > 50).mean():.1%}')
print(f'  ARRIVAL RATE:            {np.mean(arrivals):.1%} ({sum(arrivals)}/30)')
print(f'  Mean episode reward:     {np.mean(rewards):.2f}')
print(f'  Mean steps taken:        {np.mean(steps_taken):.1f}')

## 8. Copy results into the local repo clone and push

**Run this only after confirming Cells 6-7 look reasonable.**

In [ ]:
import shutil

LOCAL_CHECKPOINT_DIR = "/content/uav_trajectory_rl/checkpoints/ppo_run1"
os.makedirs(LOCAL_CHECKPOINT_DIR, exist_ok=True)

for f in glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*"):
    shutil.copy(f, LOCAL_CHECKPOINT_DIR)

print('Copied files:')
!ls -la "{LOCAL_CHECKPOINT_DIR}" 

In [ ]:
%cd /content/uav_trajectory_rl
!git config user.email "krishnasikheriya001@gmail.com"
!git config user.name "Krishna200608"

# Ensure remote URL has the token for non-interactive push
import os
if github_token:
    os.system(f"git remote set-url origin {repo_url}")

!git add -f checkpoints/ppo_run1
!git status
!git commit -m "Add PPO (M12) full training run and checkpoints"
!git push origin main
!git log --oneline -n 3

## Done

Bring the results (Step 7's evaluation numbers, and the pushed commit hash) back to the main conversation for independent review before this is used in M14's comparison plots.